# Task 3 - daily pipeline with model

This notebook builds a first full version step by step:
1. read `data.csv` in pandas chunks (file is about 11 GB)
2. aggregate 5-minute rows to `deviceId + day`
3. run rolling time validation on labelled months
4. train final model and generate monthly submission (`deviceId,year,month,prediction`)

Why daily first: monthly labels give only 7 training points per device, while daily aggregation gives much more signal and still maps cleanly to monthly submission.

In [ ]:
# Optional dependency install. Remove leading # and run once if needed.
# import sys, subprocess
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'scikit-learn', 'matplotlib'])

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error


def find_data_dir() -> Path:
    for candidate in [Path('.'), Path('task3')]:
        if (candidate / 'data.csv').exists():
            return candidate
    raise FileNotFoundError('data.csv not found in current directory or task3/')


DATA_DIR = find_data_dir()
CSV_PATH = DATA_DIR / 'data.csv'
OUT_DIR = DATA_DIR / 'out'
OUT_DIR.mkdir(exist_ok=True)

SENSOR_COLS = [f't{i}' for i in range(1, 14)]
BASE_NUM_COLS = SENSOR_COLS + ['x1']

print('DATA_DIR:', DATA_DIR.resolve())
print('CSV_PATH:', CSV_PATH.resolve())

DATA_DIR: /home/mikolaj/work/tasks-2026/task3
CSV_PATH: /home/mikolaj/work/tasks-2026/task3/data.csv


## Step 1: Build daily dataset from 5-minute data

We stream the CSV in chunks and aggregate each chunk to `deviceId + date`.

For each base column (`t1..t13`, `x1`) we keep enough stats to compute daily:
- mean
- std
- min
- max

Target on labelled period:
- `target_x2 = mean(x2)` per `deviceId + date`

Target on forecast period is naturally missing (`NaN`), which is expected.

In [2]:
def aggregate_chunk(chunk: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    ts = pd.to_datetime(chunk['timedate'], utc=True, errors='coerce')
    chunk = chunk.loc[ts.notna()].copy()
    chunk['date'] = ts.loc[ts.notna()].dt.tz_convert(None).dt.floor('D')

    chunk['active'] = (chunk['x1'] > 0).astype('float32')
    chunk['x2_filled'] = chunk['x2'].fillna(0.0).astype('float32')
    chunk['x2_count'] = chunk['x2'].notna().astype('int16')

    for col in base_cols:
        chunk[f'{col}_sq'] = chunk[col] * chunk[col]

    agg_spec = {
        'period': ('period', 'first'),
        'deviceType': ('deviceType', 'max'),
        'x3': ('x3', 'max'),
        'n_rows': ('timedate', 'size'),
        'active_sum': ('active', 'sum'),
        'x2_sum': ('x2_filled', 'sum'),
        'x2_count': ('x2_count', 'sum'),
    }
    for col in base_cols:
        agg_spec[f'{col}_sum'] = (col, 'sum')
        agg_spec[f'{col}_sq_sum'] = (f'{col}_sq', 'sum')
        agg_spec[f'{col}_min'] = (col, 'min')
        agg_spec[f'{col}_max'] = (col, 'max')

    return chunk.groupby(['deviceId', 'date'], as_index=False).agg(**agg_spec)


def collapse_partial_rows(frames: list[pd.DataFrame], base_cols: list[str]) -> pd.DataFrame:
    stacked = pd.concat(frames, ignore_index=True)

    agg_spec = {
        'period': ('period', 'first'),
        'deviceType': ('deviceType', 'max'),
        'x3': ('x3', 'max'),
        'n_rows': ('n_rows', 'sum'),
        'active_sum': ('active_sum', 'sum'),
        'x2_sum': ('x2_sum', 'sum'),
        'x2_count': ('x2_count', 'sum'),
    }
    for col in base_cols:
        agg_spec[f'{col}_sum'] = (f'{col}_sum', 'sum')
        agg_spec[f'{col}_sq_sum'] = (f'{col}_sq_sum', 'sum')
        agg_spec[f'{col}_min'] = (f'{col}_min', 'min')
        agg_spec[f'{col}_max'] = (f'{col}_max', 'max')

    return stacked.groupby(['deviceId', 'date'], as_index=False).agg(**agg_spec)


def finalize_daily_features(collapsed: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    daily = collapsed.copy()

    daily['active_ratio'] = daily['active_sum'] / daily['n_rows']
    daily['target_x2'] = daily['x2_sum'] / daily['x2_count'].replace({0: np.nan})

    for col in base_cols:
        mean_col = f'{col}_mean'
        sq_mean_col = f'{col}_sq_sum'

        daily[mean_col] = daily[f'{col}_sum'] / daily['n_rows']
        var = (daily[sq_mean_col] / daily['n_rows']) - (daily[mean_col] ** 2)
        daily[f'{col}_std'] = np.sqrt(np.clip(var, 0.0, None))

    daily['year'] = daily['date'].dt.year.astype('int16')
    daily['month'] = daily['date'].dt.month.astype('int8')
    daily['dayofyear'] = daily['date'].dt.dayofyear.astype('int16')
    daily['weekday'] = daily['date'].dt.weekday.astype('int8')
    daily['is_weekend'] = (daily['weekday'] >= 5).astype('int8')

    angle = 2.0 * np.pi * daily['dayofyear'] / 366.0
    daily['doy_sin'] = np.sin(angle)
    daily['doy_cos'] = np.cos(angle)

    daily['delta_t2_t1'] = daily['t2_mean'] - daily['t1_mean']
    daily['delta_t5_t3'] = daily['t5_mean'] - daily['t3_mean']
    daily['delta_t6_t4'] = daily['t6_mean'] - daily['t4_mean']
    daily['delta_t7_t2'] = daily['t7_mean'] - daily['t2_mean']
    daily['delta_t9_t1'] = daily['t9_mean'] - daily['t1_mean']

    stat_cols: list[str] = []
    for col in base_cols:
        stat_cols.extend([f'{col}_mean', f'{col}_std', f'{col}_min', f'{col}_max'])

    engineered_cols = [
        'delta_t2_t1',
        'delta_t5_t3',
        'delta_t6_t4',
        'delta_t7_t2',
        'delta_t9_t1',
    ]

    keep_cols = [
        'deviceId',
        'date',
        'period',
        'year',
        'month',
        'dayofyear',
        'weekday',
        'is_weekend',
        'doy_sin',
        'doy_cos',
        'deviceType',
        'x3',
        'n_rows',
        'active_ratio',
    ] + stat_cols + engineered_cols + ['target_x2']

    daily = daily[keep_cols].sort_values(['deviceId', 'date']).reset_index(drop=True)
    return daily


def build_daily_features(
    csv_path: Path,
    chunksize: int = 300_000,
    flush_every: int = 20,
    max_chunks: int | None = None,
) -> pd.DataFrame:
    usecols = ['deviceId', 'timedate', 'period', 'x3', 'deviceType'] + BASE_NUM_COLS + ['x2']

    dtype = {
        'deviceId': 'string',
        'period': 'category',
        'x3': 'Int16',
        'deviceType': 'Int16',
    }
    for col in BASE_NUM_COLS + ['x2']:
        dtype[col] = 'float32'

    partial_frames: list[pd.DataFrame] = []

    reader = pd.read_csv(
        csv_path,
        usecols=usecols,
        dtype=dtype,
        chunksize=chunksize,
        low_memory=False,
    )

    for idx, chunk in enumerate(reader, start=1):
        partial_frames.append(aggregate_chunk(chunk, BASE_NUM_COLS))

        if idx % 5 == 0:
            print(f'Processed chunks: {idx}')

        if len(partial_frames) >= flush_every:
            partial_frames = [collapse_partial_rows(partial_frames, BASE_NUM_COLS)]
            print(f'Collapsed intermediate partials at chunk {idx}')

        if max_chunks is not None and idx >= max_chunks:
            print(f'Stopping early at chunk {idx} because max_chunks={max_chunks}')
            break

    if not partial_frames:
        raise RuntimeError('No data was read from CSV.')

    collapsed = (
        collapse_partial_rows(partial_frames, BASE_NUM_COLS)
        if len(partial_frames) > 1
        else partial_frames[0]
    )

    return finalize_daily_features(collapsed, BASE_NUM_COLS)

In [3]:
DAILY_PATH = OUT_DIR / 'daily_features.csv'

if DAILY_PATH.exists():
    daily = pd.read_csv(DAILY_PATH, parse_dates=['date'])
    print(f'Loaded cached daily dataset from {DAILY_PATH.resolve()}')
else:
    # Set max_chunks to a small value (for example 4) only for quick local smoke checks.
    daily = build_daily_features(CSV_PATH, chunksize=300_000, flush_every=20, max_chunks=None)
    daily.to_csv(DAILY_PATH, index=False)
    print(f'Saved daily dataset to {DAILY_PATH.resolve()}')

print('daily shape:', daily.shape)
display(daily.head())

daily_summary = (
    daily.groupby(['period', 'year', 'month'], dropna=False)
    .agg(device_count=('deviceId', 'nunique'), row_count=('deviceId', 'size'))
    .reset_index()
    .sort_values(['year', 'month'])
)
display(daily_summary)

Processed chunks: 5
Processed chunks: 10
Processed chunks: 15
Processed chunks: 20
Collapsed intermediate partials at chunk 20
Processed chunks: 25
Processed chunks: 30
Processed chunks: 35
Collapsed intermediate partials at chunk 39
Processed chunks: 40
Processed chunks: 45
Processed chunks: 50
Processed chunks: 55
Collapsed intermediate partials at chunk 58
Processed chunks: 60
Processed chunks: 65
Processed chunks: 70
Processed chunks: 75
Collapsed intermediate partials at chunk 77
Processed chunks: 80
Processed chunks: 85
Processed chunks: 90
Processed chunks: 95
Collapsed intermediate partials at chunk 96
Processed chunks: 100
Processed chunks: 105
Processed chunks: 110
Processed chunks: 115
Collapsed intermediate partials at chunk 115
Processed chunks: 120
Processed chunks: 125
Processed chunks: 130
Collapsed intermediate partials at chunk 134
Processed chunks: 135
Processed chunks: 140
Processed chunks: 145
Processed chunks: 150
Collapsed intermediate partials at chunk 153
Proce

,deviceId,date,period,year,month,dayofyear,weekday,is_weekend,doy_sin,doy_cos,...,x1_mean,x1_std,x1_min,x1_max,delta_t2_t1,delta_t5_t3,delta_t6_t4,delta_t7_t2,delta_t9_t1,target_x2
0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01,train,2024,10,275,1,0,-0.999963,0.008583,...,0.0,0.0,0.0,0.0,-0.257118,0.442396,0.037396,0.156076,0.075833,0.102980
1,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-02,train,2024,10,276,2,0,-0.999668,0.025748,...,0.0,0.0,0.0,0.0,-0.257639,0.446528,0.046667,0.157917,0.078785,0.102154
2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-03,train,2024,10,277,3,0,-0.999079,0.042905,...,0.0,0.0,0.0,0.0,-0.253993,0.448715,0.048333,0.156736,0.086076,0.110116
3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-04,train,2024,10,278,4,0,-0.998195,0.060049,...,0.0,0.0,0.0,0.0,-0.252544,0.457951,0.046643,0.159399,0.096961,0.123136
4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-05,train,2024,10,279,5,1,-0.997018,0.077175,...,0.0,0.0,0.0,0.0,-0.250000,0.457431,0.045972,0.155833,0.096215,0.129621


,period,year,month,device_count,row_count
4,train,2024,10,596,17948
5,train,2024,11,591,17302
6,train,2024,12,590,17766
7,train,2025,1,592,17918
8,train,2025,2,588,16101
9,train,2025,3,590,17800
10,train,2025,4,595,17475
11,valid,2025,5,598,18137
12,valid,2025,6,598,17383
0,test,2025,7,591,17762


## Step 2: Prepare train/forecast sets and feature matrix

Labelled rows: daily rows where `target_x2` exists (train period).
Forecast rows: daily rows where `target_x2` is missing (valid/test periods).

We keep `period` only for diagnostics, not as a model feature.

In [ ]:
labelled = daily[daily['target_x2'].notna()].copy()
forecast = daily[daily['target_x2'].isna()].copy()

labelled['ym'] = labelled['year'] * 100 + labelled['month']
forecast['ym'] = forecast['year'] * 100 + forecast['month']

excluded_cols = {'deviceId', 'date', 'period', 'target_x2', 'ym'}
feature_cols = [c for c in daily.columns if c not in excluded_cols]

print('labelled rows:', len(labelled))
print('forecast rows:', len(forecast))
print('number of features:', len(feature_cols))
print('first 12 features:', feature_cols[:12])

In [ ]:
def make_model() -> HistGradientBoostingRegressor:
    return HistGradientBoostingRegressor(
        loss='absolute_error',
        learning_rate=0.05,
        max_depth=6,
        max_iter=400,
        min_samples_leaf=30,
        random_state=42,
    )


def prepare_X(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    X = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X.astype('float32')


def evaluate_rolling_months(
    labelled_df: pd.DataFrame,
    cols: list[str],
    first_valid_ym: int = 202501,
) -> pd.DataFrame:
    valid_months = sorted(m for m in labelled_df['ym'].unique() if m >= first_valid_ym)
    rows = []

    for valid_ym in valid_months:
        train_df = labelled_df[labelled_df['ym'] < valid_ym].copy()
        valid_df = labelled_df[labelled_df['ym'] == valid_ym].copy()

        if train_df.empty or valid_df.empty:
            continue

        device_mean = train_df.groupby('deviceId')['target_x2'].mean()
        global_mean = train_df['target_x2'].mean()

        train_base = train_df['deviceId'].map(device_mean).fillna(global_mean)
        valid_base = valid_df['deviceId'].map(device_mean).fillna(global_mean)

        baseline_pred = valid_base
        baseline_mae = mean_absolute_error(valid_df['target_x2'], baseline_pred)

        y_train_residual = train_df['target_x2'] - train_base

        model = make_model()
        model.fit(prepare_X(train_df, cols), y_train_residual)
        valid_residual = model.predict(prepare_X(valid_df, cols))

        valid_pred = (valid_base + valid_residual).clip(lower=0.0)
        model_mae = mean_absolute_error(valid_df['target_x2'], valid_pred)

        rows.append(
            {
                'valid_ym': int(valid_ym),
                'train_rows': len(train_df),
                'valid_rows': len(valid_df),
                'baseline_mae': baseline_mae,
                'model_mae': model_mae,
            }
        )

    return pd.DataFrame(rows)

In [ ]:
cv_results = evaluate_rolling_months(labelled, feature_cols, first_valid_ym=202501)
display(cv_results)

if not cv_results.empty:
    print('Average baseline MAE:', cv_results['baseline_mae'].mean())
    print('Average model MAE:   ', cv_results['model_mae'].mean())
    print('Model better on all folds:', bool((cv_results['model_mae'] < cv_results['baseline_mae']).all()))

## Step 3: Train on all labelled days and create final submission

Final strategy:
- baseline = per-device mean target on labelled period
- model learns residual over that baseline
- daily predictions are averaged to month level for submission

In [ ]:
device_mean_all = labelled.groupby('deviceId')['target_x2'].mean()
global_mean_all = labelled['target_x2'].mean()

train_base_all = labelled['deviceId'].map(device_mean_all).fillna(global_mean_all)
y_train_residual_all = labelled['target_x2'] - train_base_all

final_model = make_model()
final_model.fit(prepare_X(labelled, feature_cols), y_train_residual_all)

forecast_base = forecast['deviceId'].map(device_mean_all).fillna(global_mean_all)
forecast_residual = final_model.predict(prepare_X(forecast, feature_cols))

daily_predictions = forecast[['deviceId', 'date', 'year', 'month']].copy()
daily_predictions['prediction_daily'] = (forecast_base + forecast_residual).clip(lower=0.0)

monthly_submission = (
    daily_predictions.groupby(['deviceId', 'year', 'month'], as_index=False)['prediction_daily']
    .mean()
    .rename(columns={'prediction_daily': 'prediction'})
)

months_table = pd.DataFrame({'year': [2025] * 6, 'month': [5, 6, 7, 8, 9, 10]})
devices_table = pd.DataFrame({'deviceId': forecast['deviceId'].unique()})
target_grid = devices_table.assign(_k=1).merge(months_table.assign(_k=1), on='_k').drop(columns='_k')

monthly_submission = target_grid.merge(monthly_submission, on=['deviceId', 'year', 'month'], how='left')
fallback = monthly_submission['deviceId'].map(device_mean_all).fillna(global_mean_all)
monthly_submission['prediction'] = monthly_submission['prediction'].fillna(fallback).clip(lower=0.0)

monthly_submission = monthly_submission.sort_values(['deviceId', 'year', 'month']).reset_index(drop=True)

submission_path = OUT_DIR / 'submission_daily_hgb.csv'
monthly_submission.to_csv(submission_path, index=False)

print('Submission path:', submission_path.resolve())
print('Submission rows:', len(monthly_submission))
display(monthly_submission.head(12))

In [ ]:
expected_rows = forecast['deviceId'].nunique() * 6
print('unique forecast devices:', forecast['deviceId'].nunique())
print('expected rows (devices * 6 months):', expected_rows)
print('actual submission rows:', len(monthly_submission))
print('missing predictions:', int(monthly_submission['prediction'].isna().sum()))
print('months present:', sorted(monthly_submission['month'].unique().tolist()))

assert len(monthly_submission) == expected_rows
assert monthly_submission['prediction'].isna().sum() == 0
assert sorted(monthly_submission['month'].unique().tolist()) == [5, 6, 7, 8, 9, 10]
print('Sanity checks passed.')